In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/funnykids
/kaggle/input/datasets/funnykids/faceshifter-frame32
/kaggle/input/datasets/funnykids/faceshifter-frame32/original_frames32
/kaggle/input/datasets/funnykids/faceshifter-frame32/FaceShifter_frame32


In [2]:


# =========================================================
# DeepFake Detection - Hybrid ViT + CNN
# TRAIN: 600 IMAGES
# TEST: 100 IMAGES
# FULL FIXED VERSION
# =========================================================

# =========================================================
# INSTALLATIONS
# =========================================================
!pip uninstall -y torch torchvision torchaudio -q

!pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 \
--index-url https://download.pytorch.org/whl/cu118

!pip install -q timm==0.9.12
!pip install -q opencv-python

# =========================================================
# IMPORTS
# =========================================================
import os
import cv2
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms
from torchvision.models import resnet18

import timm

# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

# =========================================================
# DATA PATHS
# =========================================================
REAL_PATH = "/kaggle/input/datasets/funnykids/faceshifter-frame32/original_frames32"

FAKE_PATH = "/kaggle/input/datasets/funnykids/faceshifter-frame32/FaceShifter_frame32"

# =========================================================
# IMAGE SETTINGS
# =========================================================
IMG_SIZE = 224

# =========================================================
# DATA SETTINGS
# =========================================================
# TOTAL TRAIN = 600
# TOTAL TEST = 100
# TOTAL DATA = 700

TRAIN_IMAGES = 600
TEST_IMAGES = 100
TOTAL_IMAGES = TRAIN_IMAGES + TEST_IMAGES

# HALF REAL / HALF FAKE
REAL_COUNT = TOTAL_IMAGES // 2
FAKE_COUNT = TOTAL_IMAGES // 2

# =========================================================
# LOAD DATA
# =========================================================
images = []
labels = []

# =========================================================
# LOAD REAL IMAGES
# =========================================================
real_files = os.listdir(REAL_PATH)

print("\nLoading REAL images...")

for file in tqdm(real_files[:REAL_COUNT]):

    file_path = os.path.join(REAL_PATH, file)

    img = cv2.imread(file_path)

    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)

    labels.append(0)

# =========================================================
# LOAD FAKE IMAGES
# =========================================================
fake_files = os.listdir(FAKE_PATH)

print("\nLoading FAKE images...")

for file in tqdm(fake_files[:FAKE_COUNT]):

    file_path = os.path.join(FAKE_PATH, file)

    img = cv2.imread(file_path)

    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)

    labels.append(1)

# =========================================================
# CONVERT TO NUMPY
# =========================================================
images = np.array(images)

labels = np.array(labels)

print("\n========================")
print("TOTAL IMAGES:", len(images))
print("========================")

# =========================================================
# TRAIN / TEST SPLIT
# 600 TRAIN
# 100 TEST
# =========================================================
test_ratio = TEST_IMAGES / TOTAL_IMAGES

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=test_ratio,
    random_state=42,
    stratify=labels
)

print("TRAIN IMAGES:", len(X_train))
print("TEST IMAGES:", len(X_test))

# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.9, 1.0)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

# =========================================================
# DATASET
# =========================================================
class DeepFakeDataset(Dataset):

    def __init__(self, images, labels, transform=None):

        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):

        return len(self.images)

    def __getitem__(self, idx):

        image = self.images[idx]

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# =========================================================
# DATALOADER
# =========================================================
BATCH_SIZE = 8

train_dataset = DeepFakeDataset(
    X_train,
    y_train,
    train_transform
)

test_dataset = DeepFakeDataset(
    X_test,
    y_test,
    test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# =========================================================
# HYBRID MODEL
# =========================================================
class HybridModel(nn.Module):

    def __init__(self):

        super(HybridModel, self).__init__()

        # =================================================
        # CNN
        # =================================================
        self.cnn = resnet18(weights="DEFAULT")

        self.cnn.fc = nn.Identity()

        # =================================================
        # ViT
        # =================================================
        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )

        # =================================================
        # FEATURE DIMENSIONS
        # =================================================
        cnn_features = 512
        vit_features = 768

        # =================================================
        # CLASSIFIER
        # =================================================
        self.classifier = nn.Sequential(

            nn.Linear(cnn_features + vit_features, 512),

            nn.BatchNorm1d(512),

            nn.ReLU(),

            nn.Dropout(0.4),

            nn.Linear(512, 128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, 2)
        )

    def forward(self, x):

        cnn_feat = self.cnn(x)

        vit_feat = self.vit(x)

        combined = torch.cat(
            (cnn_feat, vit_feat),
            dim=1
        )

        output = self.classifier(combined)

        return output

# =========================================================
# MODEL
# =========================================================
model = HybridModel().to(device)

# =========================================================
# LOSS + OPTIMIZER
# =========================================================
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# =========================================================
# TRAINING
# =========================================================
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    predictions_train = []

    labels_train = []

    loop = tqdm(train_loader)

    for images_batch, labels_batch in loop:

        images_batch = images_batch.to(device)

        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = model(images_batch)

        loss = criterion(outputs, labels_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        predictions_train.extend(
            preds.cpu().numpy()
        )

        labels_train.extend(
            labels_batch.cpu().numpy()
        )

        acc = accuracy_score(
            labels_train,
            predictions_train
        )

        loop.set_description(
            f"Epoch [{epoch+1}/{EPOCHS}]"
        )

        loop.set_postfix(
            loss=loss.item(),
            accuracy=acc
        )

    epoch_acc = accuracy_score(
        labels_train,
        predictions_train
    )

    print("\n========================")
    print(f"EPOCH {epoch+1}")
    print("========================")

    print(
        "TRAIN LOSS:",
        running_loss / len(train_loader)
    )

    print(
        "TRAIN ACCURACY:",
        epoch_acc
    )

# =========================================================
# EVALUATION
# =========================================================
model.eval()

predictions = []

real_labels = []

with torch.no_grad():

    for images_batch, labels_batch in tqdm(test_loader):

        images_batch = images_batch.to(device)

        labels_batch = labels_batch.to(device)

        outputs = model(images_batch)

        _, preds = torch.max(outputs, 1)

        predictions.extend(
            preds.cpu().numpy()
        )

        real_labels.extend(
            labels_batch.cpu().numpy()
        )

# =========================================================
# RESULTS
# =========================================================
acc = accuracy_score(
    real_labels,
    predictions
)

print("\n========================")
print("TEST ACCURACY:", acc)
print("========================\n")

print(
    classification_report(
        real_labels,
        predictions
    )
)

# =========================================================
# CONFUSION MATRIX
# =========================================================
cm = confusion_matrix(
    real_labels,
    predictions
)

print("CONFUSION MATRIX:")
print(cm)

# =========================================================
# SAVE MODEL
# =========================================================
torch.save(
    model.state_dict(),
    "hybrid_vit_cnn_model.pth"
)

print("\nMODEL SAVED SUCCESSFULLY") 

# =========================================================
# DeepFake Detection - LSTM Model
# TRAIN: 600 IMAGES
# TEST: 100 IMAGES
# FULL FIXED VERSION
# =========================================================

# =========================================================
# INSTALLATIONS
# =========================================================
!pip uninstall -y torch torchvision torchaudio -q

!pip install -q torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 \
--index-url https://download.pytorch.org/whl/cu118

!pip install -q opencv-python

# =========================================================
# IMPORTS
# =========================================================
import os
import cv2
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms

# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

# =========================================================
# DATA PATHS
# =========================================================
REAL_PATH = "/kaggle/input/datasets/funnykids/faceshifter-frame32/original_frames32"

FAKE_PATH = "/kaggle/input/datasets/funnykids/faceshifter-frame32/FaceShifter_frame32"

# =========================================================
# IMAGE SETTINGS
# =========================================================
IMG_SIZE = 64

# =========================================================
# DATA SETTINGS
# =========================================================
TRAIN_IMAGES = 600
TEST_IMAGES = 100
TOTAL_IMAGES = TRAIN_IMAGES + TEST_IMAGES

REAL_COUNT = TOTAL_IMAGES // 2
FAKE_COUNT = TOTAL_IMAGES // 2

# =========================================================
# LOAD DATA
# =========================================================
images = []
labels = []

# =========================================================
# LOAD REAL IMAGES
# =========================================================
real_files = os.listdir(REAL_PATH)

print("\nLoading REAL images...")

for file in tqdm(real_files[:REAL_COUNT]):

    file_path = os.path.join(REAL_PATH, file)

    img = cv2.imread(file_path)

    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)

    labels.append(0)

# =========================================================
# LOAD FAKE IMAGES
# =========================================================
fake_files = os.listdir(FAKE_PATH)

print("\nLoading FAKE images...")

for file in tqdm(fake_files[:FAKE_COUNT]):

    file_path = os.path.join(FAKE_PATH, file)

    img = cv2.imread(file_path)

    if img is None:
        continue

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    images.append(img)

    labels.append(1)

# =========================================================
# CONVERT TO NUMPY
# =========================================================
images = np.array(images)

labels = np.array(labels)

print("\n========================")
print("TOTAL IMAGES:", len(images))
print("========================")

# =========================================================
# TRAIN / TEST SPLIT
# =========================================================
test_ratio = TEST_IMAGES / TOTAL_IMAGES

X_train, X_test, y_train, y_test = train_test_split(
    images,
    labels,
    test_size=test_ratio,
    random_state=42,
    stratify=labels
)

print("TRAIN IMAGES:", len(X_train))
print("TEST IMAGES:", len(X_test))

# =========================================================
# TRANSFORMS
# =========================================================
train_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

# =========================================================
# DATASET
# =========================================================
class DeepFakeDataset(Dataset):

    def __init__(self, images, labels, transform=None):

        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):

        return len(self.images)

    def __getitem__(self, idx):

        image = self.images[idx]

        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# =========================================================
# DATALOADER
# =========================================================
BATCH_SIZE = 8

train_dataset = DeepFakeDataset(
    X_train,
    y_train,
    train_transform
)

test_dataset = DeepFakeDataset(
    X_test,
    y_test,
    test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

# =========================================================
# LSTM MODEL
# =========================================================
class DeepFakeLSTM(nn.Module):

    def __init__(self):

        super(DeepFakeLSTM, self).__init__()

        # =================================================
        # IMAGE FEATURES
        # =================================================
        self.input_size = IMG_SIZE * 3

        # =================================================
        # LSTM
        # =================================================
        self.lstm = nn.LSTM(
            input_size=self.input_size,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )

        # =================================================
        # CLASSIFIER
        # =================================================
        self.classifier = nn.Sequential(

            nn.Linear(128 * 2, 128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128, 64),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(64, 2)
        )

    def forward(self, x):

        # =============================================
        # x shape:
        # [BATCH, CHANNEL, HEIGHT, WIDTH]
        # =============================================

        batch_size = x.size(0)

        # =============================================
        # RESHAPE FOR LSTM
        # =============================================
        x = x.permute(0, 2, 3, 1)

        # [BATCH, HEIGHT, WIDTH, CHANNEL]

        x = x.reshape(
            batch_size,
            IMG_SIZE,
            IMG_SIZE * 3
        )

        # =============================================
        # LSTM
        # =============================================
        lstm_out, (hidden, cell) = self.lstm(x)

        # =============================================
        # LAST OUTPUT
        # =============================================
        out = lstm_out[:, -1, :]

        # =============================================
        # CLASSIFIER
        # =============================================
        out = self.classifier(out)

        return out

# =========================================================
# MODEL
# =========================================================
model = DeepFakeLSTM().to(device)

print(model)

# =========================================================
# LOSS + OPTIMIZER
# =========================================================
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

# =========================================================
# TRAINING
# =========================================================
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0

    predictions_train = []

    labels_train = []

    loop = tqdm(train_loader)

    for images_batch, labels_batch in loop:

        images_batch = images_batch.to(device)

        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        outputs = model(images_batch)

        loss = criterion(outputs, labels_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)

        predictions_train.extend(
            preds.cpu().numpy()
        )

        labels_train.extend(
            labels_batch.cpu().numpy()
        )

        acc = accuracy_score(
            labels_train,
            predictions_train
        )

        loop.set_description(
            f"Epoch [{epoch+1}/{EPOCHS}]"
        )

        loop.set_postfix(
            loss=loss.item(),
            accuracy=acc
        )

    epoch_acc = accuracy_score(
        labels_train,
        predictions_train
    )

    print("\n========================")
    print(f"EPOCH {epoch+1}")
    print("========================")

    print(
        "TRAIN LOSS:",
        running_loss / len(train_loader)
    )

    print(
        "TRAIN ACCURACY:",
        epoch_acc
    )

# =========================================================
# EVALUATION
# =========================================================
model.eval()

predictions = []

real_labels = []

with torch.no_grad():

    for images_batch, labels_batch in tqdm(test_loader):

        images_batch = images_batch.to(device)

        labels_batch = labels_batch.to(device)

        outputs = model(images_batch)

        _, preds = torch.max(outputs, 1)

        predictions.extend(
            preds.cpu().numpy()
        )

        real_labels.extend(
            labels_batch.cpu().numpy()
        )

# =========================================================
# RESULTS
# =========================================================
acc = accuracy_score(
    real_labels,
    predictions
)

print("\n========================")
print("TEST ACCURACY:", acc)
print("========================\n")

print(
    classification_report(
        real_labels,
        predictions
    )
)

# =========================================================
# CONFUSION MATRIX
# =========================================================
cm = confusion_matrix(
    real_labels,
    predictions
)

print("CONFUSION MATRIX:")
print(cm)

# =========================================================
# SAVE MODEL
# =========================================================
torch.save(
    model.state_dict(),
    "deepfake_lstm_model.pth"
)

print("\nMODEL SAVED SUCCESSFULLY")

ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.2
DEVICE: cpu

Loading REAL images...


100%|██████████| 350/350 [00:01<00:00, 286.13it/s]



Loading FAKE images...


100%|██████████| 350/350 [00:01<00:00, 294.08it/s]



TOTAL IMAGES: 700
TRAIN IMAGES: 600
TEST IMAGES: 100


...


EPOCH 9
TRAIN LOSS: 0.6900064436594645
TRAIN ACCURACY: 0.5433333333333333


Epoch [10/10]: 100%|██████████| 75/75 [00:05<00:00, 14.67it/s, accuracy=0.543, loss=0.722]



EPOCH 10
TRAIN LOSS: 0.6893822717666626
TRAIN ACCURACY: 0.5433333333333333


100%|██████████| 13/13 [00:00<00:00, 20.57it/s]



TEST ACCURACY: 0.47

              precision    recall  f1-score   support

           0       0.47      0.40      0.43        50
           1       0.47      0.54      0.50        50

    accuracy                           0.47       100
   macro avg       0.47      0.47      0.47       100
weighted avg       0.47      0.47      0.47       100

CONFUSION MATRIX:
[[20 30]
 [23 27]]

MODEL SAVED SUCCESSFULLY
